# Sentiment Model Load & Inference Test

Evaluates fine-tuned sentiment models on **FinancialPhraseBank** (75% agreement).

| #   | Section                                          |
| --- | ------------------------------------------------ |
| 1   | Environment Check                                |
| 2   | Load FinancialPhraseBank & Split 60 / 20 / 20    |
| 3   | Shared Helpers                                   |
| 4   | Load FinBERT Test                                |
| 5   | Load LLaMA 3.2-3B-Instruct                       |
| 6   | Load Qwen 2.5-3B-Instruct (Fine-tuned)           |
| 7   | Summary — FinBERT / LLaMA / Qwen (latency table) |
| 8   | Disagreement Analysis                            |
| 9   | FinGPT via Groq API (llama-3.3-70b-versatile)    |


## 1. Environment Check


In [28]:
import subprocess

try:
    r = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
    print(r.stdout or r.stderr)
except FileNotFoundError:
    print("nvidia-smi not found")

nvidia-smi not found


python(44079) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


In [29]:
import gc, io, re, random
from pathlib import Path
import numpy as np
import pandas as pd
import requests
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    matthews_corrcoef,
)

SEED = 42
LABELS = ["negative", "neutral", "positive"]
LABEL_SET = set(LABELS)
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL = {i: l for l, i in LABEL2ID.items()}

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else ("mps" if torch.backends.mps.is_available() else "cpu")
)
DTYPE = (
    torch.bfloat16
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    else torch.float16
)
print(f"Device: {DEVICE} | DType: {DTYPE}")

Device: mps | DType: torch.float16


## 2. Load FinancialPhraseBank & Split 60 / 20 / 20


In [30]:
FPB_URL = (
    "https://raw.githubusercontent.com/maxwellsarpong/"
    "NLP-financial-text-processing-dataset/master/Sentences_75Agree.txt"
)
resp = requests.get(FPB_URL, timeout=60)
resp.raise_for_status()
fpb_df = pd.read_csv(
    io.StringIO(resp.text),
    sep="@",
    header=None,
    names=["sentence", "label_text"],
    engine="python",
    encoding="latin-1",
)
fpb_df["sentence"] = fpb_df["sentence"].str.strip()
fpb_df["label_text"] = fpb_df["label_text"].str.strip().str.lower()
fpb_df = fpb_df[fpb_df["label_text"].isin(LABEL_SET)].reset_index(drop=True)
fpb_df["label"] = fpb_df["label_text"].map(LABEL2ID)

print(f"FPB total: {len(fpb_df)} samples")
print(fpb_df["label_text"].value_counts())

train_val_df, test_df = train_test_split(
    fpb_df, test_size=0.20, random_state=SEED, stratify=fpb_df["label_text"]
)
train_df, val_df = train_test_split(
    train_val_df, test_size=0.25, random_state=SEED, stratify=train_val_df["label_text"]
)
print(f"\nTrain {len(train_df)} | Val {len(val_df)} | Test {len(test_df)}")

FPB total: 3453 samples
label_text
neutral     2146
positive     887
negative     420
Name: count, dtype: int64

Train 2071 | Val 691 | Test 691


## 3. Shared Helpers


In [31]:
def evaluate_predictions(true_labels, pred_labels):
    true_labels = [x.strip().lower() for x in true_labels]
    pred_labels = [x if x in LABEL_SET else "neutral" for x in pred_labels]
    return {
        "macro_f1": f1_score(
            true_labels, pred_labels, labels=LABELS, average="macro", zero_division=0
        ),
        "weighted_f1": f1_score(
            true_labels, pred_labels, labels=LABELS, average="weighted", zero_division=0
        ),
        "accuracy": accuracy_score(true_labels, pred_labels),
        "mcc": matthews_corrcoef(true_labels, pred_labels),
        "f1_negative": f1_score(
            true_labels,
            pred_labels,
            labels=["negative"],
            average="macro",
            zero_division=0,
        ),
        "f1_neutral": f1_score(
            true_labels,
            pred_labels,
            labels=["neutral"],
            average="macro",
            zero_division=0,
        ),
        "f1_positive": f1_score(
            true_labels,
            pred_labels,
            labels=["positive"],
            average="macro",
            zero_division=0,
        ),
        "report": classification_report(
            true_labels, pred_labels, labels=LABELS, zero_division=0
        ),
    }


def print_metrics(title, m):
    sep = "=" * 70
    print(sep, title, sep, m["report"], sep="\n")
    for k in (
        "macro_f1",
        "weighted_f1",
        "accuracy",
        "mcc",
        "f1_negative",
        "f1_neutral",
        "f1_positive",
    ):
        print(f"  {k}: {m[k]:.4f}")

## 4. Load FinBERT Test


In [32]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

FINBERT_ID = "ProsusAI/finbert"
print(f"Loading FinBERT from HuggingFace ({FINBERT_ID}) …")
finbert_tok = AutoTokenizer.from_pretrained(FINBERT_ID, use_fast=True)
finbert_model = AutoModelForSequenceClassification.from_pretrained(FINBERT_ID).to(
    DEVICE
)
finbert_model.eval()
print(f"Loaded. id2label: {finbert_model.config.id2label}")

Loading FinBERT from HuggingFace (ProsusAI/finbert) …


Loaded. id2label: {0: 'positive', 1: 'negative', 2: 'neutral'}


In [33]:
import time


def remap_finbert_pred(model, p):
    label = model.config.id2label.get(p, "neutral").lower()
    return label if label in LABEL_SET else "neutral"


print("Running FinBERT inference on TEST set …")
sents = test_df["sentence"].tolist()
finbert_preds_raw = []

_t0 = time.perf_counter()
with torch.inference_mode():
    for i in range(0, len(sents), 128):
        enc = finbert_tok(
            sents[i : i + 128],
            truncation=True,
            padding=True,
            max_length=128,
            return_tensors="pt",
        ).to(DEVICE)
        logits = finbert_model(**enc).logits
        finbert_preds_raw.extend(logits.argmax(-1).tolist())
_t1 = time.perf_counter()
finbert_latency_ms = (_t1 - _t0) * 1000 / len(sents)
print(
    f"FinBERT latency: {finbert_latency_ms:.2f} ms/sample  ({(_t1 - _t0):.1f}s total)"
)

finbert_preds = [remap_finbert_pred(finbert_model, p) for p in finbert_preds_raw]
finbert_metrics = evaluate_predictions(test_df["label_text"].tolist(), finbert_preds)
print_metrics("FinBERT — FPB Test Set", finbert_metrics)

Running FinBERT inference on TEST set …
FinBERT latency: 19.71 ms/sample  (13.6s total)
FinBERT — FPB Test Set
              precision    recall  f1-score   support

    negative       0.89      0.98      0.93        84
     neutral       1.00      0.95      0.97       429
    positive       0.90      0.97      0.93       178

    accuracy                           0.96       691
   macro avg       0.93      0.96      0.94       691
weighted avg       0.96      0.96      0.96       691

  macro_f1: 0.9447
  weighted_f1: 0.9557
  accuracy: 0.9551
  mcc: 0.9192
  f1_negative: 0.9318
  f1_neutral: 0.9701
  f1_positive: 0.9322


In [34]:
del finbert_model, finbert_tok
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("FinBERT unloaded.")

FinBERT unloaded.


## 5. Load LLaMA 3.2-3B-Instruct


In [35]:
LLAMA_DIR = Path("experiments/sentiment_agent/saved_models/llama_3.2_3b_instruct_saved")
shard_files = sorted(LLAMA_DIR.glob("model-*-of-*.safetensors"))
single_file = list(LLAMA_DIR.glob("model.safetensors"))
crdownload = list(LLAMA_DIR.glob("*.crdownload"))

if crdownload:
    print(f"WARNING: {len(crdownload)} incomplete download(s) detected:")
    for f in crdownload:
        print(f"  {f.name}  ({f.stat().st_size / 1e9:.2f} GB)")
if shard_files:
    print(f"Found {len(shard_files)} shard(s):")
    for f in shard_files:
        print(f"  {f.name}  ({f.stat().st_size / 1e9:.2f} GB)")
elif single_file:
    print(
        f"Found single model file: {single_file[0].name}  ({single_file[0].stat().st_size / 1e9:.2f} GB)"
    )
else:
    print("No safetensor model files found — check downloads.")
print(f"Model directory: {LLAMA_DIR.resolve()}")

Found 2 shard(s):
  model-00001-of-00002.safetensors  (4.97 GB)
  model-00002-of-00002.safetensors  (1.46 GB)
Model directory: /Users/lunlun/Downloads/Github/IS469_wealth_manager_project/experiments/sentiment_agent/saved_models/llama_3.2_3b_instruct_saved


In [36]:
MAX_SEQ_LEN = 512
# Keep as Path — transformers accepts Path objects and bypasses hub repo-ID string validation
LLAMA_DIR_ABS = LLAMA_DIR.resolve()

if not LLAMA_DIR_ABS.exists():
    raise FileNotFoundError(f"LLaMA model directory not found: {LLAMA_DIR_ABS}")

try:
    from unsloth import FastLanguageModel

    print("Loading with Unsloth FastLanguageModel …")
    llama_model, llama_tok = FastLanguageModel.from_pretrained(
        model_name=str(LLAMA_DIR_ABS),
        max_seq_length=MAX_SEQ_LEN,
        dtype=None,
        load_in_4bit=False,
    )
    FastLanguageModel.for_inference(llama_model)
    print("Unsloth load successful.")
except (ImportError, ModuleNotFoundError, NotImplementedError):
    from transformers import AutoModelForCausalLM, AutoTokenizer

    print("Unsloth not available — falling back to AutoModelForCausalLM …")
    # Pass Path object, not str, to avoid huggingface_hub repo-ID validation on absolute paths
    llama_tok = AutoTokenizer.from_pretrained(
        LLAMA_DIR_ABS, use_fast=True, local_files_only=True
    )
    llama_model = AutoModelForCausalLM.from_pretrained(
        LLAMA_DIR_ABS,
        torch_dtype=DTYPE,
        device_map="auto",
        local_files_only=True,
    )
    llama_model.eval()

if llama_tok.pad_token is None:
    llama_tok.pad_token = llama_tok.eos_token
llama_tok.padding_side = "left"
print(
    f"Device: {next(llama_model.parameters()).device} | Dtype: {next(llama_model.parameters()).dtype}"
)

Unsloth not available — falling back to AutoModelForCausalLM …


/var/folders/71/ljlh3xcn18gbcvg1hb44wqdh0000gn/T/ipykernel_38469/3345610295.py:9: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel
`torch_dtype` is deprecated! Use `dtype` instead!
python(44691) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

[accelerate.big_modeling|WARNING]Some parameters are on the meta device because they were offloaded to the disk.


Device: mps:0 | Dtype: torch.float16


In [37]:
def make_chat_prompt(tokenizer, sentence: str) -> str:
    return tokenizer.apply_chat_template(
        [
            {
                "role": "system",
                "content": "Financial sentiment classifier. One word reply: negative, neutral, or positive.",
            },
            {"role": "user", "content": f"Classify: {sentence}"},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )


def parse_label(text: str) -> str:
    m = re.search(r"(negative|neutral|positive)", text.strip().lower())
    return m.group(1) if m else "neutral"


@torch.inference_mode()
def batched_generate_labels(
    model, tokenizer, sentences, batch_size=16, max_new_tokens=8
):
    model.eval()
    preds = []
    for i in range(0, len(sentences), batch_size):
        batch = sentences[i : i + batch_size]
        prompts = [make_chat_prompt(tokenizer, s) for s in batch]
        enc = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=256,
        ).to(model.device)
        out = model.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
        new_tokens = out[:, enc["input_ids"].shape[1] :]
        decoded = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)
        preds.extend([parse_label(t) for t in decoded])
        if (i // batch_size) % 10 == 0:
            print(f"  [{i + len(batch)}/{len(sentences)}] done", flush=True)
    return preds


# Sanity check
sample = "The company reported record-breaking profits for the quarter."
print(f"Sanity check → '{sample}'")
print(
    f"Predicted: {batched_generate_labels(llama_model, llama_tok, [sample], batch_size=1)[0]}"
)

Sanity check → 'The company reported record-breaking profits for the quarter.'
  [1/1] done
Predicted: positive


In [38]:
import time

print("Running LLaMA inference on TEST set …")
_t0 = time.perf_counter()
llama_preds = batched_generate_labels(
    llama_model, llama_tok, test_df["sentence"].tolist(), batch_size=16
)
_t1 = time.perf_counter()
llama_latency_ms = (_t1 - _t0) * 1000 / len(test_df)
print(f"LLaMA latency: {llama_latency_ms:.2f} ms/sample  ({(_t1 - _t0):.1f}s total)")
llama_metrics = evaluate_predictions(test_df["label_text"].tolist(), llama_preds)
print_metrics("LLaMA 3.2-3B-Instruct (fine-tuned) — FPB Test Set", llama_metrics)

Running LLaMA inference on TEST set …
  [16/691] done
  [176/691] done
  [336/691] done
  [496/691] done
  [656/691] done
LLaMA latency: 805.26 ms/sample  (556.4s total)
LLaMA 3.2-3B-Instruct (fine-tuned) — FPB Test Set
              precision    recall  f1-score   support

    negative       0.96      0.95      0.96        84
     neutral       0.97      0.99      0.98       429
    positive       0.98      0.93      0.96       178

    accuracy                           0.97       691
   macro avg       0.97      0.96      0.96       691
weighted avg       0.97      0.97      0.97       691

  macro_f1: 0.9639
  weighted_f1: 0.9695
  accuracy: 0.9696
  mcc: 0.9428
  f1_negative: 0.9581
  f1_neutral: 0.9770
  f1_positive: 0.9568


In [39]:
del llama_model, llama_tok
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("LLaMA unloaded.")

LLaMA unloaded.


## 6. Load Qwen 2.5-3B-Instruct (Fine-tuned)

Loads the locally saved Qwen 2.5-3B model from `saved_models/qwen/` (sharded `.safetensors`).  
Reuses the same `batched_generate_labels` helper as LLaMA — Qwen's tokenizer also supports `apply_chat_template` with the ChatML format.


In [41]:
QWEN_DIR = Path("experiments/sentiment_agent/saved_models/qwen")
if not QWEN_DIR.exists():
    raise FileNotFoundError(
        f"Qwen model not found at {QWEN_DIR.resolve()}\n"
        "Adjust QWEN_DIR to point at the folder containing the .safetensors shards."
    )

shard_files = sorted(QWEN_DIR.glob("model-*-of-*.safetensors"))
single_file = list(QWEN_DIR.glob("model.safetensors"))
if shard_files:
    print(f"Found {len(shard_files)} shard(s):")
    for f in shard_files:
        print(f"  {f.name}  ({f.stat().st_size / 1e9:.2f} GB)")
elif single_file:
    print(
        f"Found single file: {single_file[0].name}  ({single_file[0].stat().st_size / 1e9:.2f} GB)"
    )
else:
    raise FileNotFoundError("No .safetensors files found — check QWEN_DIR")

QWEN_DIR_ABS = str(QWEN_DIR.resolve())

try:
    from unsloth import FastLanguageModel

    print("Loading Qwen with Unsloth FastLanguageModel …")
    qwen_model, qwen_tok = FastLanguageModel.from_pretrained(
        model_name=QWEN_DIR_ABS,
        max_seq_length=MAX_SEQ_LEN,
        dtype=None,
        load_in_4bit=False,
    )
    FastLanguageModel.for_inference(qwen_model)
    print("Unsloth load successful.")
except (ImportError, ModuleNotFoundError, NotImplementedError):
    from transformers import AutoModelForCausalLM, AutoTokenizer

    print("Unsloth not available — falling back to AutoModelForCausalLM …")
    qwen_tok = AutoTokenizer.from_pretrained(QWEN_DIR_ABS, use_fast=True)
    qwen_model = AutoModelForCausalLM.from_pretrained(
        QWEN_DIR_ABS,
        torch_dtype=DTYPE,
        device_map="auto",
    )
    qwen_model.eval()

if qwen_tok.pad_token is None:
    qwen_tok.pad_token = qwen_tok.eos_token
qwen_tok.padding_side = "left"
print(
    f"Device: {next(qwen_model.parameters()).device} | Dtype: {next(qwen_model.parameters()).dtype}"
)

# Sanity check
_sample = "The company reported record-breaking profits for the quarter."
print(f"\nSanity check → '{_sample}'")
print(
    f"Predicted: {batched_generate_labels(qwen_model, qwen_tok, [_sample], batch_size=1)[0]}"
)

Found 2 shard(s):
  model-00001-of-00002.safetensors  (3.97 GB)
  model-00002-of-00002.safetensors  (2.20 GB)
Unsloth not available — falling back to AutoModelForCausalLM …


/var/folders/71/ljlh3xcn18gbcvg1hb44wqdh0000gn/T/ipykernel_38469/1166696565.py:24: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

[accelerate.big_modeling|WARNING]Some parameters are on the meta device because they were offloaded to the disk.


Device: mps:0 | Dtype: torch.float16

Sanity check → 'The company reported record-breaking profits for the quarter.'
  [1/1] done
Predicted: positive


In [42]:
import time

print("Running Qwen 2.5-3B inference on TEST set …")
_t0 = time.perf_counter()
qwen_preds = batched_generate_labels(
    qwen_model, qwen_tok, test_df["sentence"].tolist(), batch_size=16
)
_t1 = time.perf_counter()
qwen_latency_ms = (_t1 - _t0) * 1000 / len(test_df)
print(f"Qwen latency: {qwen_latency_ms:.2f} ms/sample  ({(_t1 - _t0):.1f}s total)")
qwen_metrics = evaluate_predictions(test_df["label_text"].tolist(), qwen_preds)
print_metrics("Qwen 2.5-3B-Instruct (fine-tuned) — FPB Test Set", qwen_metrics)

Running Qwen 2.5-3B inference on TEST set …
  [16/691] done
  [176/691] done
  [336/691] done
  [496/691] done
  [656/691] done
Qwen latency: 971.39 ms/sample  (671.2s total)
Qwen 2.5-3B-Instruct (fine-tuned) — FPB Test Set
              precision    recall  f1-score   support

    negative       0.99      0.98      0.98        84
     neutral       0.96      0.98      0.97       429
    positive       0.95      0.92      0.94       178

    accuracy                           0.96       691
   macro avg       0.97      0.96      0.96       691
weighted avg       0.96      0.96      0.96       691

  macro_f1: 0.9634
  weighted_f1: 0.9637
  accuracy: 0.9638
  mcc: 0.9319
  f1_negative: 0.9820
  f1_neutral: 0.9711
  f1_positive: 0.9371


In [43]:
del qwen_model, qwen_tok
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Qwen unloaded.")

Qwen unloaded.


## 7. Summary — FinBERT / LLaMA / Qwen


In [44]:
cols = [
    "model",
    "macro_f1",
    "weighted_f1",
    "accuracy",
    "mcc",
    "f1_negative",
    "f1_neutral",
    "f1_positive",
    "latency_ms_per_sample",
]

rows = [
    {
        "model": "FinBERT (pretrained)",
        **{k: v for k, v in finbert_metrics.items() if isinstance(v, float)},
        "latency_ms_per_sample": finbert_latency_ms,
    },
    {
        "model": "LLaMA 3.2-3B-Instruct (fine-tuned)",
        **{k: v for k, v in llama_metrics.items() if isinstance(v, float)},
        "latency_ms_per_sample": llama_latency_ms,
    },
    {
        "model": "Qwen 2.5-3B-Instruct (fine-tuned)",
        **{k: v for k, v in qwen_metrics.items() if isinstance(v, float)},
        "latency_ms_per_sample": qwen_latency_ms,
    },
]

display(
    pd.DataFrame(rows)[cols]
    .sort_values("macro_f1", ascending=False)
    .reset_index(drop=True)
    .style.format({c: "{:.4f}" for c in cols if c != "model"})
    .highlight_max(
        subset=[c for c in cols if c not in ("model", "latency_ms_per_sample")],
        color="lightgreen",
    )
    .highlight_min(subset=["latency_ms_per_sample"], color="lightgreen")
)

,model,macro_f1,weighted_f1,accuracy,mcc,f1_negative,f1_neutral,f1_positive,latency_ms_per_sample
0,LLaMA 3.2-3B-Instruct (fine-tuned),0.9639,0.9695,0.9696,0.9428,0.9581,0.9770,0.9568,805.2551
1,Qwen 2.5-3B-Instruct (fine-tuned),0.9634,0.9637,0.9638,0.9319,0.9820,0.9711,0.9371,971.3889
2,FinBERT (pretrained),0.9447,0.9557,0.9551,0.9192,0.9318,0.9701,0.9322,19.7148


## 8. Disagreement Analysis


In [50]:
disagree_df = pd.DataFrame(
    {
        "Sentence": test_df["sentence"].tolist(),
        "Human Label": test_df["label_text"].tolist(),
        "FinBERT": finbert_preds,
        "Llama-QLoRA": llama_preds,
        "Qwen-2.5-3B": qwen_preds,
    }
)

# Rows where at least one model disagrees with the others
_model_cols = ["FinBERT", "Llama-QLoRA", "Qwen-2.5-3B"]
disagree_df = disagree_df[disagree_df[_model_cols].nunique(axis=1) > 1].reset_index(
    drop=True
)
disagree_df.index += 1
disagree_df.index.name = "#"

pd.set_option("display.max_colwidth", None)

print(
    f"Disagreements: {len(disagree_df)} / {len(test_df)} sentences ({len(disagree_df) / len(test_df) * 100:.1f}%)"
)
display(disagree_df)

Disagreements: 54 / 691 sentences (7.8%)


,Sentence,Human Label,FinBERT,Llama-QLoRA,Qwen-2.5-3B
#,,,,,
1,The new units should become one of the largest ones within the company .,neutral,positive,neutral,neutral
2,Finnish component supplier Componenta Corporation OMX Helsinki : CTH1V said on Monday 16 June that it is changing its pricing cycle due to the increase of raw material prices .,neutral,neutral,negative,neutral
3,This is the company 's first contract abroad .,neutral,positive,neutral,neutral
4,Electricity consumption grows with higher frequencies .,neutral,positive,positive,neutral
5,"Among other industrial stocks , Metso added 0.53 pct to 40.04 eur , Wartsila B was down 0.77 pct at 47.87 eur and Rautaruukki K was 1.08 pct lower at 37.56 eur .",neutral,negative,negative,neutral
6,`` The new unit is a major investment in the Finnish media scene .,neutral,neutral,neutral,positive
7,"In addition , the existing service counter area in the reception hall will be rebuilt and access provided to local rail connections .",neutral,positive,neutral,neutral
8,"Many of the commercial vessels had got stuck in the narrow Bay of Bothnia , where the ice is thicker , and around the Aaland islands .",negative,negative,neutral,negative
9,`` Directors and shareholders alike should ask why these practices were allowed to continue . '',neutral,neutral,neutral,negative


In [46]:
_model_cols = ["FinBERT", "Llama-QLoRA", "Qwen-2.5-3B"]
correct_counts = {
    col: (disagree_df[col] == disagree_df["Human Label"]).sum() for col in _model_cols
}
none_correct = (
    ~disagree_df[_model_cols].eq(disagree_df["Human Label"], axis=0).any(axis=1)
).sum()

print(f"On {len(disagree_df)} disagreements:")
for col, cnt in correct_counts.items():
    print(f"  {col} correct : {cnt}")
print(f"  None correct      : {none_correct}")

On 54 disagreements:
  FinBERT correct : 26
  Llama-QLoRA correct : 36
  Qwen-2.5-3B correct : 32
  None correct      : 0


## 9. FinGPT via Groq API (llama-3.3-70b-versatile)

Uses the Groq OpenAI-compatible endpoint. FinGPT isn't directly hosted on Groq, so we use `llama-3.3-70b-versatile` with a financial-sentiment system prompt matching the same task.

> **Note:** Groq latency includes per-request network round-trips and the mandatory 0.05 s inter-request sleep, so it is not directly comparable to local inference latency.


In [47]:
import subprocess, sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "openai"])

import time
from openai import OpenAI

GROQ_API_KEY = "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"
GROQ_MODEL = "llama-3.3-70b-versatile"

groq_client = OpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1",
)

# Quick sanity check
resp = groq_client.chat.completions.create(
    model=GROQ_MODEL,
    messages=[
        {
            "role": "system",
            "content": "Financial sentiment classifier. Reply with exactly one word: negative, neutral, or positive.",
        },
        {
            "role": "user",
            "content": "Classify: The company reported record-breaking profits for the quarter.",
        },
    ],
    max_tokens=5,
    temperature=0,
)
print("Groq client ready. Sanity check →", resp.choices[0].message.content.strip())

python(66091) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip


Groq client ready. Sanity check → Positive.


In [48]:
def groq_predict(sentence: str, retries: int = 5) -> str:
    """Single-sentence Groq inference with exponential back-off on rate limits."""
    for attempt in range(retries):
        try:
            r = groq_client.chat.completions.create(
                model=GROQ_MODEL,
                messages=[
                    {
                        "role": "system",
                        "content": "Financial sentiment classifier. Reply with exactly one word: negative, neutral, or positive.",
                    },
                    {"role": "user", "content": f"Classify: {sentence}"},
                ],
                max_tokens=5,
                temperature=0,
            )
            raw = r.choices[0].message.content.strip().lower()
            m = re.search(r"(negative|neutral|positive)", raw)
            return m.group(1) if m else "neutral"
        except Exception as e:
            if "rate" in str(e).lower() and attempt < retries - 1:
                wait = 2**attempt
                print(f"  Rate-limited, waiting {wait}s …")
                time.sleep(wait)
            else:
                print(f"  Error on attempt {attempt + 1}: {e}")
                return "neutral"
    return "neutral"


print("Running Groq inference on TEST set …")
groq_preds = []
total = len(test_df)
_t0 = time.perf_counter()
for idx, sentence in enumerate(test_df["sentence"].tolist()):
    groq_preds.append(groq_predict(sentence))
    if (idx + 1) % 50 == 0 or (idx + 1) == total:
        print(f"  [{idx + 1}/{total}] done", flush=True)
    time.sleep(0.05)  # ~20 req/s, well under Groq's free-tier limit
_t1 = time.perf_counter()
groq_latency_ms = (_t1 - _t0) * 1000 / total

print(
    f"Groq latency: {groq_latency_ms:.1f} ms/sample  ({(_t1 - _t0):.1f}s total)  [includes network + sleep]"
)
groq_metrics = evaluate_predictions(test_df["label_text"].tolist(), groq_preds)
print_metrics(f"Groq ({GROQ_MODEL}) — FPB Test Set", groq_metrics)

Running Groq inference on TEST set …
  [50/691] done
  [100/691] done
  [150/691] done
  [200/691] done
  [250/691] done
  [300/691] done
  [350/691] done
  [400/691] done
  [450/691] done
  [500/691] done
  [550/691] done
  [600/691] done
  [650/691] done
  [691/691] done
Groq latency: 2356.8 ms/sample  (1628.6s total)  [includes network + sleep]
Groq (llama-3.3-70b-versatile) — FPB Test Set
              precision    recall  f1-score   support

    negative       0.90      0.98      0.94        84
     neutral       0.99      0.58      0.73       429
    positive       0.51      0.99      0.67       178

    accuracy                           0.74       691
   macro avg       0.80      0.85      0.78       691
weighted avg       0.85      0.74      0.74       691

  macro_f1: 0.7815
  weighted_f1: 0.7428
  accuracy: 0.7366
  mcc: 0.6452
  f1_negative: 0.9371
  f1_neutral: 0.7331
  f1_positive: 0.6743


In [49]:
# Full summary including Groq (API latency is not comparable to local)
rows_full = [
    {
        "model": "FinBERT (pretrained)",
        **{k: v for k, v in finbert_metrics.items() if isinstance(v, float)},
        "latency_ms_per_sample": finbert_latency_ms,
    },
    {
        "model": "LLaMA 3.2-3B-Instruct (fine-tuned)",
        **{k: v for k, v in llama_metrics.items() if isinstance(v, float)},
        "latency_ms_per_sample": llama_latency_ms,
    },
    {
        "model": "Qwen 2.5-3B-Instruct (fine-tuned)",
        **{k: v for k, v in qwen_metrics.items() if isinstance(v, float)},
        "latency_ms_per_sample": qwen_latency_ms,
    },
    {
        "model": f"Groq {GROQ_MODEL} (API+sleep)",
        **{k: v for k, v in groq_metrics.items() if isinstance(v, float)},
        "latency_ms_per_sample": groq_latency_ms,
    },
]

display(
    pd.DataFrame(rows_full)[cols]
    .sort_values("macro_f1", ascending=False)
    .reset_index(drop=True)
    .style.format({c: "{:.4f}" for c in cols if c != "model"})
    .highlight_max(
        subset=[c for c in cols if c not in ("model", "latency_ms_per_sample")],
        color="lightgreen",
    )
    .highlight_min(subset=["latency_ms_per_sample"], color="lightgreen")
)

,model,macro_f1,weighted_f1,accuracy,mcc,f1_negative,f1_neutral,f1_positive,latency_ms_per_sample
0,LLaMA 3.2-3B-Instruct (fine-tuned),0.9639,0.9695,0.9696,0.9428,0.9581,0.9770,0.9568,805.2551
1,Qwen 2.5-3B-Instruct (fine-tuned),0.9634,0.9637,0.9638,0.9319,0.9820,0.9711,0.9371,971.3889
2,FinBERT (pretrained),0.9447,0.9557,0.9551,0.9192,0.9318,0.9701,0.9322,19.7148
3,Groq llama-3.3-70b-versatile (API+sleep),0.7815,0.7428,0.7366,0.6452,0.9371,0.7331,0.6743,2356.8255
